# DWPose (RTMPose) 微调 Notebook
## 基于 Mixamo 生成的 COCO17 姿态数据

本 notebook 包含:
1. 零样本效果展示 (预训练模型)
2. 数据准备和转换
3. 模型微调训练
4. 微调后效果对比

In [ ]:
# 安装依赖
!pip install -q torch torchvision
!pip install -q opencv-python matplotlib numpy
!pip install -q mmengine mmcv mmpose
!pip install -q rtmlib

In [ ]:
import os
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from rtmlib import PoseTracker, draw_skeleton

## 1. 零样本效果展示 - 预训练模型

In [ ]:
# 初始化预训练 DWPose 模型
pose_tracker = PoseTracker(
    det_frequency=1,
    mode='performance',
    backend='onnxruntime',
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"使用设备: {pose_tracker.device}")
print("预训练模型加载完成")

In [ ]:
# 零样本测试 - 在生成的图像上测试预训练模型
def test_pretrained_model(image_path, gt_keypoints=None):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    keypoints, scores = pose_tracker(img)
    
    fig, axes = plt.subplots(1, 2 if gt_keypoints else 1, figsize=(15, 7))
    
    if gt_keypoints is None:
        axes = [axes]
    
    # 预测结果
    img_pred = img_rgb.copy()
    if len(keypoints) > 0:
        img_pred = draw_skeleton(img_pred, keypoints[0], scores[0])
    axes[0].imshow(img_pred)
    axes[0].set_title('预训练模型预测 (零样本)')
    axes[0].axis('off')
    
    # Ground Truth
    if gt_keypoints is not None:
        img_gt = img_rgb.copy()
        img_gt = draw_skeleton(img_gt, gt_keypoints, np.ones(17))
        axes[1].imshow(img_gt)
        axes[1].set_title('Ground Truth (Mixamo)')
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return keypoints, scores

# 测试第一个视角的第一帧
test_img = '68 model/01 visual_view/01_1 Rendering/frame_0001.png'
test_json = '68 model/exported_coco17_strict1/01 visual_view/frame_0001_coco17_strict1.json'

if os.path.exists(test_img) and os.path.exists(test_json):
    with open(test_json, 'r') as f:
        gt_data = json.load(f)
    
    # 转换 GT 为 numpy 数组 (17, 2)
    gt_kpts = np.array([[v['x'], v['y']] for v in gt_data['keypoints'].values()])
    
    print("\n=== 零样本测试 ===")
    pred_kpts, pred_scores = test_pretrained_model(test_img, gt_kpts)
    
    if len(pred_kpts) > 0:
        error = np.mean(np.linalg.norm(pred_kpts[0] - gt_kpts, axis=1))
        print(f"平均关键点误差: {error:.2f} 像素")
else:
    print("测试图像或标注文件不存在")

## 2. 数据准备 - 转换为训练格式

In [ ]:
class MixamoPoseDataset(Dataset):
    def __init__(self, root_dir, views=['01', '02', '03', '04', '05'], transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        # 收集所有样本
        for view in views:
            img_dir = self.root_dir / f"{view} visual_view" / f"{view}_1 Rendering"
            json_dir = self.root_dir / "exported_coco17_strict1" / f"{view} visual_view"
            
            if not img_dir.exists() or not json_dir.exists():
                continue
            
            for json_file in sorted(json_dir.glob('frame_*_coco17_strict1.json')):
                frame_num = json_file.stem.split('_')[1]
                img_file = img_dir / f"frame_{frame_num}.png"
                
                if img_file.exists():
                    self.samples.append((str(img_file), str(json_file)))
        
        print(f"加载了 {len(self.samples)} 个训练样本")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, json_path = self.samples[idx]
        
        # 读取图像
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # 读取关键点
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        keypoints = np.array([[v['x'], v['y']] for v in data['keypoints'].values()], dtype=np.float32)
        visibility = np.ones(17, dtype=np.float32)  # 所有关键点可见
        
        # 归一化到 [0, 1]
        h, w = img.shape[:2]
        keypoints[:, 0] /= w
        keypoints[:, 1] /= h
        
        if self.transform:
            img = self.transform(img)
        else:
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        
        return {
            'image': img,
            'keypoints': torch.from_numpy(keypoints),
            'visibility': torch.from_numpy(visibility)
        }

# 创建数据集
dataset = MixamoPoseDataset('68 model')

# 划分训练集和验证集 (80/20)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

print(f"训练集: {len(train_dataset)} 样本")
print(f"验证集: {len(val_dataset)} 样本")

In [ ]:
# 可视化数据样本
sample = dataset[0]
img = sample['image'].permute(1, 2, 0).numpy()
kpts = sample['keypoints'].numpy()

# 反归一化
h, w = img.shape[:2]
kpts[:, 0] *= w
kpts[:, 1] *= h

plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.scatter(kpts[:, 0], kpts[:, 1], c='red', s=50)
for i, (x, y) in enumerate(kpts):
    plt.text(x, y, str(i), color='white', fontsize=8)
plt.title('训练数据样本')
plt.axis('off')
plt.show()

## 3. 模型微调

In [ ]:
# 简化的姿态估计模型 (基于 RTMPose 架构)
class SimplePoseModel(nn.Module):
    def __init__(self, num_keypoints=17):
        super().__init__()
        # 使用 ResNet-like backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1),
            
            self._make_layer(64, 128, 2),
            self._make_layer(128, 256, 2),
            self._make_layer(256, 512, 2),
        )
        
        # Heatmap head
        self.head = nn.Sequential(
            nn.Conv2d(512, 256, 3, 1, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_keypoints, 1)
        )
        
        # Regression head (直接回归坐标)
        self.regressor = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_keypoints * 2)
        )
    
    def _make_layer(self, in_channels, out_channels, num_blocks):
        layers = []
        layers.append(nn.Conv2d(in_channels, out_channels, 3, 2, 1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))
        
        for _ in range(num_blocks - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, 3, 1, 1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        features = self.backbone(x)
        coords = self.regressor(features).reshape(-1, 17, 2)
        return coords

# 初始化模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimplePoseModel().to(device)
print(f"模型参数量: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")

In [ ]:
# 训练配置
batch_size = 8
num_epochs = 50
learning_rate = 1e-3

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

print("训练配置完成")

In [ ]:
# 训练循环
train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # 训练
    model.train()
    train_loss = 0
    for batch in train_loader:
        images = batch['image'].to(device)
        keypoints = batch['keypoints'].to(device)
        
        optimizer.zero_grad()
        pred_keypoints = model(images)
        loss = criterion(pred_keypoints, keypoints)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)
    
    # 验证
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['image'].to(device)
            keypoints = batch['keypoints'].to(device)
            
            pred_keypoints = model(images)
            loss = criterion(pred_keypoints, keypoints)
            val_loss += loss.item()
    
    val_loss /= len(val_loader)
    val_losses.append(val_loss)
    
    scheduler.step()
    
    # 保存最佳模型
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_dwpose_finetuned.pth')
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

print(f"\n训练完成! 最佳验证损失: {best_val_loss:.6f}")

In [ ]:
# 绘制训练曲线
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('训练曲线')
plt.legend()
plt.grid(True)
plt.show()

## 4. 微调后效果对比

In [ ]:
# 加载最佳模型
model.load_state_dict(torch.load('best_dwpose_finetuned.pth'))
model.eval()

# 对比测试
def compare_models(img_path, json_path):
    # 读取图像和 GT
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    
    with open(json_path, 'r') as f:
        gt_data = json.load(f)
    gt_kpts = np.array([[v['x'], v['y']] for v in gt_data['keypoints'].values()])
    
    # 预训练模型预测
    pretrained_kpts, pretrained_scores = pose_tracker(img)
    
    # 微调模型预测
    img_tensor = torch.from_numpy(img_rgb).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    img_tensor = img_tensor.to(device)
    
    with torch.no_grad():
        finetuned_kpts = model(img_tensor)[0].cpu().numpy()
    
    # 反归一化
    finetuned_kpts[:, 0] *= w
    finetuned_kpts[:, 1] *= h
    
    # 可视化对比
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    # 预训练模型
    img1 = img_rgb.copy()
    if len(pretrained_kpts) > 0:
        img1 = draw_skeleton(img1, pretrained_kpts[0], pretrained_scores[0])
        error1 = np.mean(np.linalg.norm(pretrained_kpts[0] - gt_kpts, axis=1))
    else:
        error1 = float('inf')
    axes[0].imshow(img1)
    axes[0].set_title(f'预训练模型\n误差: {error1:.2f}px')
    axes[0].axis('off')
    
    # 微调模型
    img2 = img_rgb.copy()
    img2 = draw_skeleton(img2, finetuned_kpts, np.ones(17))
    error2 = np.mean(np.linalg.norm(finetuned_kpts - gt_kpts, axis=1))
    axes[1].imshow(img2)
    axes[1].set_title(f'微调模型\n误差: {error2:.2f}px')
    axes[1].axis('off')
    
    # Ground Truth
    img3 = img_rgb.copy()
    img3 = draw_skeleton(img3, gt_kpts, np.ones(17))
    axes[2].imshow(img3)
    axes[2].set_title('Ground Truth')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return error1, error2

# 测试多个样本
test_samples = [
    ('68 model/01 visual_view/01_1 Rendering/frame_0001.png', 
     '68 model/exported_coco17_strict1/01 visual_view/frame_0001_coco17_strict1.json'),
    ('68 model/02 visual_view/02_1 Rendering/frame_0020.png',
     '68 model/exported_coco17_strict1/02 visual_view/frame_0020_coco17_strict1.json'),
    ('68 model/03 visual_view/03_1 Rendering/frame_0040.png',
     '68 model/exported_coco17_strict1/03 visual_view/frame_0040_coco17_strict1.json'),
]

pretrained_errors = []
finetuned_errors = []

for img_path, json_path in test_samples:
    if os.path.exists(img_path) and os.path.exists(json_path):
        print(f"\n测试: {os.path.basename(img_path)}")
        e1, e2 = compare_models(img_path, json_path)
        pretrained_errors.append(e1)
        finetuned_errors.append(e2)

# 统计结果
print("\n=== 整体对比 ===")
print(f"预训练模型平均误差: {np.mean(pretrained_errors):.2f}px")
print(f"微调模型平均误差: {np.mean(finetuned_errors):.2f}px")
print(f"改进: {(1 - np.mean(finetuned_errors)/np.mean(pretrained_errors))*100:.1f}%")

## 5. 保存和导出

In [ ]:
# 保存最终模型
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_losses': train_losses,
    'val_losses': val_losses,
    'best_val_loss': best_val_loss,
}, 'dwpose_finetuned_checkpoint.pth')

print("模型已保存到: dwpose_finetuned_checkpoint.pth")
print("\n微调完成!")